# L8.3 — Agent Memory: short-term, long-term, hybrid

Hands-on notebook for the lesson [`8-3-memory.mdx`](../../llm-quest-theory/level-8/8-3-memory.mdx).

> **Learning objectives**
> - Build all 4 classical memory stores: **buffer**, **sliding window**, **summarisation**, and **vector long-term memory**.
> - Combine them into a **hybrid** memory that answers a question 50 turns later about a fact stated at turn 3.
> - Attach structured key-value memory for user-profile facts ("my name is Minh").
> - Isolate memory per `user_id` and demonstrate that user A cannot read user B's memory.

## Connection to the theory
Covers **§1–§14** of the source `.mdx`. Everything runs locally with `sentence-transformers`.

In [1]:
# ---- Setup ----
import os, uuid, json, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("embedding dim:", encoder.get_sentence_embedding_dimension())

embedding dim: 384


## 1. Buffer memory — keep everything
Simple list of messages. Fine for short chats, breaks for long ones.

In [2]:
class BufferMemory:
    def __init__(self):
        self.messages = []
    def add(self, role, content):
        self.messages.append({"role": role, "content": content, "t": time.time()})
    def context(self):
        return list(self.messages)

bm = BufferMemory()
bm.add("user", "hi"); bm.add("assistant", "hello"); bm.add("user", "my name is Minh")
print("buffer size:", len(bm.context()))

buffer size: 3


## 2. Sliding window — keep the last N turns

In [3]:
class SlidingMemory:
    def __init__(self, max_messages=20):
        self.messages = []
        self.max = max_messages
    def add(self, role, content):
        self.messages.append({"role": role, "content": content, "t": time.time()})
    def context(self):
        # Always preserve the very first message (system prompt, if any)
        head = self.messages[:1] if self.messages and self.messages[0]["role"] == "system" else []
        return head + self.messages[-self.max:]

sm = SlidingMemory(max_messages=3)
sm.add("system", "You are an assistant.")
for t in range(10):
    sm.add("user", f"turn {t}")
print("kept messages (with system head):",
      [m["content"] for m in sm.context()])

kept messages (with system head): ['You are an assistant.', 'turn 7', 'turn 8', 'turn 9']


## 3. Summarisation memory — compress old turns
We swap the oldest half of the buffer for a one-line summary. In production this summary comes from an LLM; here we use a deterministic string to keep the notebook reproducible.

In [4]:
class SummarisingMemory:
    def __init__(self, max_messages=10, summariser=None):
        self.messages = []
        self.max = max_messages
        self.summary = ""
        self.summariser = summariser or self._default_summarise

    @staticmethod
    def _default_summarise(old):
        user_lines      = [m["content"] for m in old if m["role"] == "user"]
        assistant_lines = [m["content"] for m in old if m["role"] == "assistant"]
        return (f"Earlier: the user said {len(user_lines)} things, the assistant replied {len(assistant_lines)} times. "
                f"Recent user lines: {user_lines[-3:]}")

    def add(self, role, content):
        self.messages.append({"role": role, "content": content, "t": time.time()})
        if len(self.messages) > self.max:
            half = len(self.messages) // 2
            self.summary = self.summariser(self.messages[:half])
            self.messages = self.messages[half:]

    def context(self):
        head = [{"role": "system", "content": f"Context so far: {self.summary}"}] if self.summary else []
        return head + self.messages

smem = SummarisingMemory(max_messages=6)
for i in range(12):
    smem.add("user", f"message #{i}")
    smem.add("assistant", f"ack #{i}")
print("summary:", smem.summary)
print("live messages:", [m["content"] for m in smem.messages])

summary: Earlier: the user said 1 things, the assistant replied 2 times. Recent user lines: ['message #8']
live messages: ['message #9', 'ack #9', 'message #10', 'ack #10', 'message #11', 'ack #11']


## 4. Vector long-term memory
Save every "fact-worthy" line with its embedding, recall by cosine similarity. This is the memory store behind "the agent still remembers what you said a week ago".

In [5]:
class VectorMemory:
    def __init__(self, encoder):
        self.encoder = encoder
        self.entries = []        # list of dicts
        self.embs    = np.zeros((0, encoder.get_sentence_embedding_dimension()), dtype=np.float32)

    def save(self, content, user_id="default", meta=None):
        vec = self.encoder.encode([content], convert_to_numpy=True, normalize_embeddings=True)
        self.entries.append({
            "id":      str(uuid.uuid4())[:8],
            "user_id": user_id,
            "content": content,
            "meta":    meta or {},
            "t":       time.time(),
        })
        self.embs = np.vstack([self.embs, vec])

    def recall(self, query, user_id="default", k=3):
        if len(self.entries) == 0: return []
        # Restrict to the user's own rows
        own_idx = [i for i, e in enumerate(self.entries) if e["user_id"] == user_id]
        if not own_idx: return []
        q = self.encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
        sims = (self.embs[own_idx] @ q)
        order = np.argsort(-sims)[:k]
        return [(self.entries[own_idx[i]], float(sims[i])) for i in order]

vm = VectorMemory(encoder)
vm.save("User's favourite drink is bubble tea",   user_id="u1", meta={"type": "preference"})
vm.save("User's birthday is March 15",            user_id="u1", meta={"type": "fact"})
vm.save("User prefers responses in Vietnamese",   user_id="u1", meta={"type": "preference"})
vm.save("Another user says they like espresso",   user_id="u2")

hits = vm.recall("When is the user's birthday?", user_id="u1", k=2)
for e, s in hits:
    print(f"  {s:.3f}  {e['content']}")

  0.830  User's birthday is March 15
  0.222  User's favourite drink is bubble tea


## 5. Structured key-value memory
Exact lookup for "what's my name?" questions that don't deserve vector search.

In [6]:
class ProfileMemory:
    def __init__(self):
        self.store = {}   # user_id -> dict[key, value]
    def set(self, user_id, key, value):
        self.store.setdefault(user_id, {})[key] = value
    def get(self, user_id, key):
        return self.store.get(user_id, {}).get(key)
    def all(self, user_id):
        return dict(self.store.get(user_id, {}))

prof = ProfileMemory()
prof.set("u1", "name",     "Minh")
prof.set("u1", "language", "vi")
prof.set("u2", "name",     "Alex")
print("u1 profile:", prof.all("u1"))
print("u2 profile:", prof.all("u2"))

u1 profile: {'name': 'Minh', 'language': 'vi'}
u2 profile: {'name': 'Alex'}


## 6. Hybrid memory — the production pattern
The agent, at each turn, assembles the prompt from:
- a **system** line,
- the user's **profile** (key-value),
- **retrieved episodes** from the vector store,
- the **recent** sliding window.

New user messages get both appended to the sliding window AND saved to the vector store if they look like facts.

In [7]:
class HybridAgentMemory:
    def __init__(self, encoder, window=6):
        self.window  = SlidingMemory(max_messages=window)
        self.long    = VectorMemory(encoder)
        self.profile = ProfileMemory()

    def ingest_user(self, user_id, text):
        self.window.add("user", text)
        # Tiny extractor: pull "my name is X" into profile; pull otherwise-facty sentences into LTM.
        t = text.lower()
        if "my name is" in t:
            name = text.split("my name is", 1)[1].strip().rstrip(".")
            self.profile.set(user_id, "name", name.split()[0])
        if any(k in t for k in ["my birthday", "my favourite", "i love", "i like", "i prefer", "i use"]):
            self.long.save(text, user_id=user_id, meta={"source": "user_fact"})

    def ingest_assistant(self, user_id, text):
        self.window.add("assistant", text)

    def build_prompt(self, user_id, user_message, k=3):
        parts = ["You are an assistant with memory."]
        prof = self.profile.all(user_id)
        if prof:
            parts.append("User profile: " + json.dumps(prof, ensure_ascii=False))
        hits = self.long.recall(user_message, user_id=user_id, k=k)
        if hits:
            parts.append("Relevant memory:")
            for e, s in hits:
                parts.append(f"  - ({s:.2f}) {e['content']}")
        parts.append("\nRecent conversation:")
        for m in self.window.context():
            parts.append(f"  {m['role']}: {m['content']}")
        parts.append(f"  user: {user_message}")
        return "\n".join(parts)

ham = HybridAgentMemory(encoder, window=6)

## 7. The 50-turn simulation
At turn 3, user says their name and birthday. We fill ~45 turns of filler chat. At turn 50 we ask about the birthday — the hybrid memory should retrieve the original fact.

In [8]:
USER = "u1"

# Turn 1-2: small talk
for pair in [("hi", "Hello!"), ("nice weather today", "Indeed!")]:
    ham.ingest_user(USER, pair[0])
    ham.ingest_assistant(USER, pair[1])

# Turn 3: important facts
ham.ingest_user(USER, "By the way, my name is Minh and my birthday is March 15.")
ham.ingest_assistant(USER, "Happy to meet you, Minh!")

# Turns 4-49: filler
for i in range(4, 50):
    ham.ingest_user(USER, f"Can you help me with task #{i}?")
    ham.ingest_assistant(USER, f"Done with task #{i}.")

# Turn 50: the question
prompt = ham.build_prompt(USER, "What day is my birthday?")
print(prompt)

You are an assistant with memory.
User profile: {"name": "Minh"}
Relevant memory:
  - (0.62) By the way, my name is Minh and my birthday is March 15.

Recent conversation:
  user: Can you help me with task #47?
  assistant: Done with task #47.
  user: Can you help me with task #48?
  assistant: Done with task #48.
  user: Can you help me with task #49?
  assistant: Done with task #49.
  user: What day is my birthday?


The 45 filler turns pushed the original fact out of the sliding window — but the vector store returned it right at the top of "Relevant memory". The agent now has enough context to answer correctly.

## 8. Profile was also extracted
The name got parsed into `ProfileMemory`. An LLM-based extractor would do the same job more robustly.

In [9]:
print("profile for u1:", ham.profile.all(USER))

profile for u1: {'name': 'Minh'}


## 9. Per-user isolation check
User `u1` should never see facts saved under `u2`. The filter lives inside `VectorMemory.recall`.

In [10]:
# Save a secret into u2
ham.long.save("Secret: user 2's account recovery code is ALPHA-42", user_id="u2", meta={"source": "user_fact"})

# u1 asks about it
hits = ham.long.recall("What is my account recovery code?", user_id="u1", k=3)
print("u1 sees:", hits)

# u2 asks the same
hits2 = ham.long.recall("What is my account recovery code?", user_id="u2", k=3)
print("u2 sees:", [(e["content"], s) for e, s in hits2])

u1 sees: [({'id': '24c83968', 'user_id': 'u1', 'content': 'By the way, my name is Minh and my birthday is March 15.', 'meta': {'source': 'user_fact'}, 't': 1776769858.9432058}, 0.11784573644399643)]
u2 sees: [("Secret: user 2's account recovery code is ALPHA-42", 0.6419013738632202)]


## 10. Consolidation — a TTL pass
Production systems run periodic jobs that drop stale entries. We simulate a TTL filter: delete everything older than `ttl_seconds`.

In [11]:
def consolidate(vm: VectorMemory, ttl_seconds: float):
    """Drop entries older than `ttl_seconds`. Returns count removed."""
    now = time.time()
    keep_idx = [i for i, e in enumerate(vm.entries) if now - e["t"] < ttl_seconds]
    removed = len(vm.entries) - len(keep_idx)
    vm.entries = [vm.entries[i] for i in keep_idx]
    vm.embs    = vm.embs[keep_idx] if keep_idx else np.zeros((0, vm.embs.shape[1]), dtype=np.float32)
    return removed

# Inject a synthetic "old" entry
ham.long.save("User moved to Da Nang (two years ago)", user_id="u1")
ham.long.entries[-1]["t"] -= 365 * 24 * 3600 * 2   # backdate by two years
before = len(ham.long.entries)
removed = consolidate(ham.long, ttl_seconds=365 * 24 * 3600)   # TTL = 1 year
print(f"entries: {before} before  {len(ham.long.entries)} after  ({removed} removed)")

entries: 3 before  2 after  (1 removed)


## 11. Quick checks

In [12]:
# Sliding window never exceeds `max`
sm = SlidingMemory(max_messages=5)
for i in range(50): sm.add("user", str(i))
assert len([m for m in sm.context() if m["role"] != "system"]) <= 5
# Summariser kicks in when the buffer exceeds `max`
assert smem.summary, "summariser should have run"
# Vector recall honours user_id isolation
u1_hits = ham.long.recall("recovery code", user_id="u1", k=3)
assert all(e["user_id"] == "u1" for e, _ in u1_hits)
# Profile extractor caught the name
assert ham.profile.all("u1").get("name") == "Minh"
# The 50-turn prompt contains a reference to March 15 via the vector store
assert "march 15" in prompt.lower() or "march15" in prompt.lower()
# Consolidation kept the non-stale entries
assert all("two years ago" not in e["content"] for e in ham.long.entries)
print("OK — all four memory patterns work, hybrid survives the 50-turn test.")

OK — all four memory patterns work, hybrid survives the 50-turn test.


## Reflection questions

1. Our fact extractor uses hand-written keyword triggers. Name two failure modes and describe the LLM-based version you'd deploy instead.
2. Vector memory returns items from *any* past turn. That is great for recall but risky for privacy — you might surface something the user later asked to forget. How would you support a "GDPR delete my data" button?
3. The sliding window in the hybrid agent is 6 turns. For a customer-support bot that mixes 30-turn conversations with quick one-shot queries, how would you choose the window size dynamically?
4. Consolidation here is a TTL. List two *semantic* consolidation passes the theory mentions (merging duplicates, resolving stale entries) and sketch how you'd implement one.

## References
- Source theory: [`8-3-memory.mdx`](../../llm-quest-theory/level-8/8-3-memory.mdx)
- Next: [`8-4-multi-agent`](8-4-multi-agent.ipynb)